# Notebook 02: Preprocessing & Feature Engineering

## Pipeline
1. Load raw CSV data for all 12 cities
2. Handle missing values (linear interpolation + forward fill)
3. Engineer 34 features (temporal, solar geometry, meteorological, geographic)
4. Filter daylight hours (solar zenith < 85°)
5. Apply feature-aware normalization (Z-score for continuous, skip binary/cyclic)
6. Create sliding windows (72h context → 24h forecast)
7. Train/Val/Test split by date (no leakage)
8. Save processed data as Parquet + numpy windows

In [1]:
import sys, os
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))

from utils.city_config import (
    CITY_METADATA, get_training_cities, get_test_cities, get_all_cities,
    CONTEXT_LENGTH, FORECAST_HORIZON, STRIDE,
    TRAIN_END, VAL_END, TEST_END
)
from utils.feature_engineering import (
    engineer_all_features, filter_daylight_hours, get_feature_columns,
    create_sliding_windows
)
from utils.normalization import (
    GlobalZScoreScaler, apply_feature_normalization, get_normalizable_features
)

print(f"Context: {CONTEXT_LENGTH}h, Forecast: {FORECAST_HORIZON}h, Stride: {STRIDE}h")
print(f"Feature columns: {len(get_feature_columns())}")
print(f"Splits: Train ≤ {TRAIN_END} | Val ≤ {VAL_END} | Test ≤ {TEST_END}")

Context: 72h, Forecast: 24h, Stride: 24h
Feature columns: 34
Splits: Train ≤ 2021-12-31 | Val ≤ 2022-06-30 | Test ≤ 2024-06-30


## 1. Load Raw Data & Engineer Features

In [2]:
def load_and_process_city(city_name, raw_dir):
    """Load raw CSV and apply full feature engineering pipeline."""
    # Find CSV
    csv_files = list(Path(raw_dir).glob(f'{city_name}_raw_*.csv'))
    if not csv_files:
        print(f"  WARNING: No CSV found for {city_name} in {raw_dir}")
        return None
    
    df = pd.read_csv(csv_files[0], parse_dates=['timestamp'])
    print(f"  [{city_name}] Loaded {len(df):,} rows from {csv_files[0].name}")
    
    # Handle missing values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    # Replace NASA fill values (-999) with NaN
    for col in numeric_cols:
        df[col] = df[col].replace(-999, np.nan)
    
    # Interpolate (linear for short gaps, forward fill for longer)
    df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit=6)
    df[numeric_cols] = df[numeric_cols].ffill(limit=24)
    df[numeric_cols] = df[numeric_cols].bfill(limit=24)
    
    missing_after = df[numeric_cols].isna().sum().sum()
    print(f"  [{city_name}] Remaining NaNs after interpolation: {missing_after}")
    
    # Engineer all 34 features
    df = engineer_all_features(df, city_name)
    print(f"  [{city_name}] Features engineered: {df.shape[1]} columns")
    
    return df

# Process all cities
all_city_data = {}

print("=" * 60)
print(" Processing TRAINING cities")
print("=" * 60)
for city in get_training_cities():
    df = load_and_process_city(city, 'data/raw')
    if df is not None:
        all_city_data[city] = df

print("\n" + "=" * 60)
print(" Processing TEST cities")
print("=" * 60)
for city in get_test_cities():
    df = load_and_process_city(city, 'data/raw_unseen')
    if df is not None:
        all_city_data[city] = df

print(f"\nTotal processed: {len(all_city_data)} cities")

 Processing TRAINING cities
  [jaipur] Loaded 92,016 rows from jaipur_raw_2013_2024.csv
  [jaipur] Remaining NaNs after interpolation: 0
  [jaipur] Features engineered: 40 columns
  [delhi] Loaded 100,776 rows from delhi_raw_2013_2024.csv
  [delhi] Remaining NaNs after interpolation: 0
  [delhi] Features engineered: 40 columns
  [guwahati] Loaded 100,776 rows from guwahati_raw_2013_2024.csv
  [guwahati] Remaining NaNs after interpolation: 0
  [guwahati] Features engineered: 40 columns
  [ahmedabad] Loaded 100,776 rows from ahmedabad_raw_2013_2024.csv
  [ahmedabad] Remaining NaNs after interpolation: 0
  [ahmedabad] Features engineered: 40 columns
  [chennai] Loaded 100,776 rows from chennai_raw_2013_2024.csv
  [chennai] Remaining NaNs after interpolation: 0
  [chennai] Features engineered: 40 columns
  [thiruvananthapuram] Loaded 100,776 rows from thiruvananthapuram_raw_2013_2024.csv
  [thiruvananthapuram] Remaining NaNs after interpolation: 0
  [thiruvananthapuram] Features engineered

## 2. Filter Daylight Hours

In [3]:
# Filter daylight hours (solar zenith < 85°)
daylight_data = {}
for city, df in all_city_data.items():
    df_day = filter_daylight_hours(df, zenith_threshold=85.0)
    pct = len(df_day) / len(df) * 100
    daylight_data[city] = df_day
    print(f"  {city:25s}: {len(df):>8,} -> {len(df_day):>8,} daylight rows ({pct:.1f}%)")

print(f"\nNote: We keep ALL hours (including night) for sliding windows.")
print(f"Daylight filtering is used only for evaluation metrics.")

  jaipur                   :   92,016 ->   43,085 daylight rows (46.8%)
  delhi                    :  100,776 ->   47,148 daylight rows (46.8%)
  guwahati                 :  100,776 ->   47,246 daylight rows (46.9%)
  ahmedabad                :  100,776 ->   47,317 daylight rows (47.0%)
  chennai                  :  100,776 ->   47,734 daylight rows (47.4%)
  thiruvananthapuram       :  100,776 ->   46,687 daylight rows (46.3%)
  nagpur                   :  100,776 ->   47,466 daylight rows (47.1%)
  hyderabad                :  100,776 ->   47,211 daylight rows (46.8%)
  bangalore                :  100,776 ->   46,850 daylight rows (46.5%)
  kolkata                  :  100,776 ->   47,300 daylight rows (46.9%)
  lucknow                  :  100,776 ->   47,449 daylight rows (47.1%)
  jodhpur                  :  100,776 ->   47,180 daylight rows (46.8%)

Note: We keep ALL hours (including night) for sliding windows.
Daylight filtering is used only for evaluation metrics.


## 3. Train/Val/Test Split

In [4]:
# Split by date
train_end = pd.Timestamp(TRAIN_END)
val_end = pd.Timestamp(VAL_END)

city_splits = {}
print(f"{'City':25s} {'Train':>10s} {'Val':>10s} {'Test':>10s}")
print("-" * 60)

for city, df in all_city_data.items():
    train = df[df['timestamp'] <= train_end].copy()
    val = df[(df['timestamp'] > train_end) & (df['timestamp'] <= val_end)].copy()
    test = df[df['timestamp'] > val_end].copy()
    
    city_splits[city] = {'train': train, 'val': val, 'test': test}
    print(f"  {city:25s} {len(train):>10,} {len(val):>10,} {len(test):>10,}")

City                           Train        Val       Test
------------------------------------------------------------
  jaipur                        70,099      4,344     17,573
  delhi                         78,859      4,344     17,573
  guwahati                      78,859      4,344     17,573
  ahmedabad                     78,859      4,344     17,573
  chennai                       78,859      4,344     17,573
  thiruvananthapuram            78,859      4,344     17,573
  nagpur                        78,859      4,344     17,573
  hyderabad                     78,859      4,344     17,573
  bangalore                     78,859      4,344     17,573
  kolkata                       78,859      4,344     17,573
  lucknow                       78,859      4,344     17,573
  jodhpur                       78,859      4,344     17,573


## 4. Normalization

Strategy:
- Z-score normalize continuous meteorological/radiation features
- Leave binary, cyclic, and geographic features unchanged
- Fit scaler on TRAINING data only (no data leakage)

In [6]:
feature_cols = get_feature_columns()
normalizable = get_normalizable_features(feature_cols)

print(f"Total features: {len(feature_cols)}")
print(f"Z-score normalized: {len(normalizable)}")
print(f"Kept as-is: {len(feature_cols) - len(normalizable)}")
print(f"\nNormalized features: {normalizable}")

# Fit scaler on combined training data from ALL training cities
train_frames = []
for city in get_training_cities():
    if city in city_splits:
        train_df = city_splits[city]['train']
        # Ensure all feature columns exist
        for col in feature_cols:
            if col not in train_df.columns:
                train_df[col] = 0.0
        train_frames.append(train_df[feature_cols].values)

combined_train = np.vstack(train_frames).astype(np.float64)
print(f"\nCombined training data shape: {combined_train.shape}, dtype: {combined_train.dtype}")

# Replace any remaining NaN/inf with 0
combined_train = np.nan_to_num(combined_train, nan=0.0, posinf=0.0, neginf=0.0)

# Fit global scaler
scaler = GlobalZScoreScaler()
normalize_mask = np.array([f in normalizable for f in feature_cols])
scaler.fit(combined_train[:, normalize_mask], 
           [feature_cols[i] for i, m in enumerate(normalize_mask) if m])

# Save scaler
os.makedirs('data/processed', exist_ok=True)
scaler.save('data/processed/normalization_stats.json')
print("Scaler stats saved to data/processed/normalization_stats.json")

Total features: 34
Z-score normalized: 18
Kept as-is: 16

Normalized features: ['ALLSKY_SFC_SW_DWN', 'CLRSKY_SFC_SW_DWN', 'T2M', 'T2M_MAX_24h', 'T2M_MIN_24h', 'RH2M', 'PRECTOTCORR', 'PS', 'WS10M', 'CLOUD_AMT', 'ALLSKY_SFC_SW_DIFF', 'CLRSKY_SFC_SW_DIFF', 'ALLSKY_SFC_UV_INDEX', 'ALLSKY_SFC_PAR_TOT', 'solar_elevation', 'day_length', 'extraterrestrial_irradiance', 'dewpoint']

Combined training data shape: (543253, 34), dtype: float64
Scaler stats saved to data/processed/normalization_stats.json


## 5. Create Sliding Windows & Save

In [7]:
# Create sliding windows for each city and split
os.makedirs('data/processed/windows', exist_ok=True)

window_stats = []

for city in get_all_cities():
    if city not in city_splits:
        continue
    
    for split_name in ['train', 'val', 'test']:
        df_split = city_splits[city][split_name]
        if len(df_split) < CONTEXT_LENGTH + FORECAST_HORIZON:
            print(f"  SKIP {city}/{split_name}: too few rows ({len(df_split)})")
            continue
        
        # Ensure all feature columns exist
        for col in feature_cols:
            if col not in df_split.columns:
                df_split[col] = 0
        
        # Create windows
        contexts, targets, timestamps = create_sliding_windows(
            df_split,
            context_length=CONTEXT_LENGTH,
            forecast_horizon=FORECAST_HORIZON,
            stride=STRIDE,
            feature_columns=feature_cols,
        )
        
        if len(contexts) == 0:
            continue
        
        # Save
        prefix = f'data/processed/windows/{city}_{split_name}'
        np.save(f'{prefix}_contexts.npy', contexts)
        np.save(f'{prefix}_targets.npy', targets)
        
        window_stats.append({
            'city': city,
            'split': split_name,
            'n_windows': len(contexts),
            'context_shape': contexts.shape,
            'target_shape': targets.shape,
        })
        print(f"  {city}/{split_name}: {len(contexts)} windows, "
              f"ctx={contexts.shape}, tgt={targets.shape}")

# Summary
ws_df = pd.DataFrame(window_stats)
print(f"\n{'='*60}")
print(f"Total windows: {ws_df['n_windows'].sum():,}")
display(ws_df.pivot_table(index='city', columns='split', values='n_windows', aggfunc='sum', fill_value=0))

  jaipur/train: 2917 windows, ctx=(2917, 72, 34), tgt=(2917, 24)
  jaipur/val: 178 windows, ctx=(178, 72, 34), tgt=(178, 24)
  jaipur/test: 729 windows, ctx=(729, 72, 34), tgt=(729, 24)
  delhi/train: 3282 windows, ctx=(3282, 72, 34), tgt=(3282, 24)
  delhi/val: 178 windows, ctx=(178, 72, 34), tgt=(178, 24)
  delhi/test: 729 windows, ctx=(729, 72, 34), tgt=(729, 24)
  guwahati/train: 3282 windows, ctx=(3282, 72, 34), tgt=(3282, 24)
  guwahati/val: 178 windows, ctx=(178, 72, 34), tgt=(178, 24)
  guwahati/test: 729 windows, ctx=(729, 72, 34), tgt=(729, 24)
  ahmedabad/train: 3282 windows, ctx=(3282, 72, 34), tgt=(3282, 24)
  ahmedabad/val: 178 windows, ctx=(178, 72, 34), tgt=(178, 24)
  ahmedabad/test: 729 windows, ctx=(729, 72, 34), tgt=(729, 24)
  chennai/train: 3282 windows, ctx=(3282, 72, 34), tgt=(3282, 24)
  chennai/val: 178 windows, ctx=(178, 72, 34), tgt=(178, 24)
  chennai/test: 729 windows, ctx=(729, 72, 34), tgt=(729, 24)
  thiruvananthapuram/train: 3282 windows, ctx=(3282, 72

split,test,train,val
city,,,
ahmedabad,729,3282,178
bangalore,729,3282,178
chennai,729,3282,178
delhi,729,3282,178
guwahati,729,3282,178
hyderabad,729,3282,178
jaipur,729,2917,178
jodhpur,729,3282,178
kolkata,729,3282,178


## 6. Save Processed DataFrames

In [8]:
# Save full processed DataFrames as Parquet for EDA
os.makedirs('data/processed/parquet', exist_ok=True)

for city, df in all_city_data.items():
    outpath = f'data/processed/parquet/{city}_processed.parquet'
    df.to_parquet(outpath, index=False)
    print(f"  {city}: saved {len(df):,} rows -> {outpath}")

print(f"\nAll processed data saved. Ready for EDA (Notebook 03) and modeling.")

  jaipur: saved 92,016 rows -> data/processed/parquet/jaipur_processed.parquet
  delhi: saved 100,776 rows -> data/processed/parquet/delhi_processed.parquet
  guwahati: saved 100,776 rows -> data/processed/parquet/guwahati_processed.parquet
  ahmedabad: saved 100,776 rows -> data/processed/parquet/ahmedabad_processed.parquet
  chennai: saved 100,776 rows -> data/processed/parquet/chennai_processed.parquet
  thiruvananthapuram: saved 100,776 rows -> data/processed/parquet/thiruvananthapuram_processed.parquet
  nagpur: saved 100,776 rows -> data/processed/parquet/nagpur_processed.parquet
  hyderabad: saved 100,776 rows -> data/processed/parquet/hyderabad_processed.parquet
  bangalore: saved 100,776 rows -> data/processed/parquet/bangalore_processed.parquet
  kolkata: saved 100,776 rows -> data/processed/parquet/kolkata_processed.parquet
  lucknow: saved 100,776 rows -> data/processed/parquet/lucknow_processed.parquet
  jodhpur: saved 100,776 rows -> data/processed/parquet/jodhpur_process

## Summary

**Processing complete:**
- 34 features engineered per city
- Sliding windows: 72h context → 24h forecast, stride=24h
- Normalization: Z-score (train-fitted) for continuous features
- Files saved: Parquet (full), NumPy windows (modeling)

**Next:** Notebook 03 — EDA & Visualization